In [1]:
import candas as can
import pathlib as pl
import numpy as np
from scipy import stats
import matplotlib as mpl

# mpl.use('Cairo')  # for saving SVGs that Affinity Designer can parse
import matplotlib.pyplot as plt
import seaborn as sns

from candas.test import QuantStudio

import gumbi as gmb

WARNING (theano.link.c.cmodule): install mkl with `conda install mkl-service`: No module named 'mkl'
/opt/anaconda3/envs/can_manuscript/lib/python3.9/site-packages/arviz/data/base.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from utils import savefig

In [3]:
code_pth = pl.Path.cwd()  # for running in Jupyter
# code_pth = pl.Path(__file__)  # for running in terminal
fig_pth = code_pth.parent
data_pth = fig_pth / 'data'

plt.style.use(str(can.style.breve))

%config InlineBackend.figure_format = 'retina'

## Global Figure Parameters

In [4]:
width = 1.625 / 2
height = 1.65 / 2
figsize = (width, height)
spotsize = 8**2
linewidth = 2
ticklabelsize = 8
labelsize = 10
titlesize = labelsize + 2

radar_figsize = (1.25, 1.25)
radar_spotsize = 8

disease, healthy = hex_color, fam_color = sns.diverging_palette(20, 220, s=100, n=2)
palette = sns.diverging_palette(20, 220, as_cmap=True)


def format_ax(
    ax,
    figsize=figsize,
    mar_l=0.56,
    mar_r=0.17,
    mar_t=0.22,
    mar_b=0.48,
    ticklabelsize=ticklabelsize,
    labelsize=labelsize,
    titlesize=titlesize,
    **kwargs,
):
    width, height = figsize

    ax.set_ylabel(ax.get_ylabel(), fontsize=labelsize)
    ax.set_xlabel(ax.get_xlabel(), fontsize=labelsize)
    ax.set_title(ax.get_title(), fontsize=titlesize)

    ax.tick_params(axis="both", length=1, width=0.5, labelsize=ticklabelsize)

    plt.subplots_adjust(
        left=mar_l / width,
        right=1 - mar_r / width,
        top=1 - mar_t / height,
        bottom=mar_b / height,
        **kwargs,
    )
    plt.setp(ax.spines.values(), linewidth=0.5)

    return ax

# Radar Plots

## Plotting functions

In [5]:
def symmetrize(x):
    return np.hstack([x, x[:, ::-1]])[:, ::2]


def circularize(x):
    return np.hstack([x, x[:, 0].reshape(-1, 1)])

In [6]:
def find_centroid(rs, label_loc):
    rs = np.expand_dims(rs, axis=-1) if len(rs.shape) == 2 else rs
    θs = np.expand_dims(label_loc, axis=(0, 2))

    # Convert from polar to cartesian coordinates
    xy = np.stack([np.cos(θs) * rs, np.sin(θs) * rs], axis=-1)
    x = np.cos(θs) * rs
    y = np.sin(θs) * rs

    # Centroid of a polygon
    A = 1 / 2 * np.sum(np.cross(xy[:, :-1], xy[:, 1:]), axis=1, keepdims=True)
    Cx = (
        1
        / (6 * A)
        * np.sum(
            (xy[:, :-1, :, 0] + xy[:, 1:, :, 0])
            * np.cross(xy[:, :-1, ...], xy[:, 1:, ...]),
            axis=1,
            keepdims=True,
        )
    )
    Cy = (
        1
        / (6 * A)
        * np.sum(
            (xy[:, :-1, :, 1] + xy[:, 1:, :, 1])
            * np.cross(xy[:, :-1, ...], xy[:, 1:, ...]),
            axis=1,
            keepdims=True,
        )
    )

    centroid_r = np.sqrt(Cx**2 + Cy**2)
    centroid_θ = np.arctan2(Cy, Cx) % (2 * np.pi)

    return centroid_θ, centroid_r

In [7]:
def prepare_radar_plot(label_loc, labels, r_max=6, figsize=None):
    figsize = figsize or (1.1, 1.1)

    fig = plt.figure(figsize=figsize)
    ax = plt.subplot(polar=True)

    _ = ax.set_thetagrids(np.degrees(label_loc), labels=labels)
    ax.set_yticklabels([])
    ax.set_ylim(0, r_max)
    ax.set_yticks(np.arange(r_max + 1))
    ax.set_theta_zero_location("N")

    return fig, ax


def rotate_radar_labels(ax, ticklabelsize=ticklabelsize, y_offset=0):
    angles = np.linspace(0, 2 * np.pi, len(ax.get_xticklabels())) % (2 * np.pi)
    angles = np.rad2deg(angles)
    labels = []
    for label, angle in zip(ax.get_xticklabels(), angles):
        x, y = label.get_position()
        lab = ax.text(
            x,
            y + y_offset,
            "RNA " + label.get_text(),
            fontsize=ticklabelsize,
            transform=label.get_transform(),
            ha=label.get_ha(),
            va=label.get_va(),
            #   bbox={'pad':0.01}
        )
        lab.set_rotation(angle)
        labels.append(lab)
    ax.set_xticklabels([])
    return labels

## Synthetic Populations

In [8]:
indiv_lw = 0.5
indiv_fill_alpha = 0.1
indiv_cent_alpha = 0.5
pop_lw = 1
pop_fill_alpha = 0.5
cent_ms = 4

In [9]:
SEED = 1
n_genes = 5
n_patients = 5
r_max = 6

# Averages for lognormal distribution
locs = np.arange(n_genes)
locs = np.vstack([locs, locs + stats.norm(-1, 1).rvs(n_genes, random_state=SEED)])

# Generate a range of population averages for each gene
pop_avgs = stats.lognorm(loc=locs, s=1.2).rvs([2, n_genes], random_state=SEED)

# Inversely sort each condition
pop_avgs = np.sort(pop_avgs, axis=1)
pop_avgs[1, :] = pop_avgs[1, ::-1]

# Symmetrize and circularize to define the final population averages
pop_avgs = circularize(symmetrize(pop_avgs))

# Rescale averages values to [0, 5]
pop_avgs = pop_avgs / pop_avgs.max() * (r_max - 1)

labels = np.arange(n_genes) + 1
labels = [*labels, labels[0]]
label_loc = np.linspace(start=0, stop=2 * np.pi, num=n_genes + 1)

# Draw samples from each population average and circularize
scales = stats.uniform(0.2, 0.5).rvs([1, n_genes], random_state=SEED)

samples = stats.lognorm(loc=pop_avgs[:, :-1] - 1, s=scales).rvs(
    [n_patients, 2, n_genes], random_state=SEED
)
samples = np.moveaxis(samples, 0, -1)
samples = np.concatenate([samples, samples[:, [0], :]], axis=1)

## Disease patients

In [12]:
fig, ax = prepare_radar_plot(label_loc, labels)

# Individual samples
ax.plot(
    label_loc,
    samples[0, ...],
    color=disease,
    lw=indiv_lw,
    # marker='o', ms=5
)
for sample in samples[0, ...].T:
    ax.fill(label_loc, sample, lw=0, color=disease, alpha=indiv_fill_alpha)

print(label_loc)
print(samples[0, ...].T)
# Individual sample centroids
# centroid_θ, centroid_r = find_centroid(samples, label_loc)
# ax.plot(centroid_θ[0, :], centroid_r[0, :], 'o', ms=cent_ms, color=disease, alpha=indiv_cent_alpha)

rotate_radar_labels(ax, y_offset=0.2)
format_ax(
    ax,
    figsize=radar_figsize,
    mar_l=0.175,
    mar_r=0.175,
    mar_t=0.175,
    mar_b=0.175,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, "miniature_bass")

[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[[1.92783013 1.89273949 4.57951128 4.23376206 1.9531203  1.92783013]
 [1.80331435 1.49824482 4.61731923 4.42153438 2.04956076 1.80331435]
 [0.62400559 3.08169307 4.87744494 4.74068781 1.96545737 0.62400559]
 [0.7399901  1.98359427 4.55134203 4.2908891  1.51855578 0.7399901 ]
 [0.91075702 1.79109763 4.54094537 5.35952426 1.70019377 0.91075702]]


<PolarAxes: >

## Healthy patients

In [13]:
fig, ax = prepare_radar_plot(label_loc, labels)

# Individual samples
ax.plot(
    label_loc,
    samples[1, ...],
    color=healthy,
    lw=indiv_lw,
    # marker='o', ms=5
)
for sample in samples[1, ...].T:
    ax.fill(label_loc, sample, lw=0, color=healthy, alpha=indiv_fill_alpha)

print(label_loc)
print(samples[1, ...].T)

# Individual sample centroids
# centroid_θ, centroid_r = find_centroid(samples, label_loc)
# ax.plot(centroid_θ[1, :], centroid_r[1, :], 'o', ms=cent_ms, color=healthy, alpha=indiv_cent_alpha)

rotate_radar_labels(ax, y_offset=0.2)
format_ax(
    ax,
    figsize=radar_figsize,
    mar_l=0.175,
    mar_r=0.175,
    mar_t=0.175,
    mar_b=0.175,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, alias="triangular_cobra")

[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[[4.3905482  3.25234179 0.31686267 0.70024149 3.00376341 4.3905482 ]
 [4.63806336 1.50279366 0.29705423 0.59662308 3.24238746 4.63806336]
 [4.75630515 1.52834092 0.28739082 0.49190469 3.22568906 4.75630515]
 [4.99483973 1.1296567  0.50613267 1.37285419 3.29456309 4.99483973]
 [4.77088218 1.707735   0.98034113 0.62478729 3.25346409 4.77088218]]


<PolarAxes: >

## Population plots

In [14]:
fig, ax = prepare_radar_plot(label_loc, labels)

ax.plot(label_loc, pop_avgs[0, :], color=disease, lw=pop_lw)
# ax.plot(label_loc, samples[0, :, :5], color=healthy, lw=1, alpha=0.5)
ax.fill(label_loc, pop_avgs[0, :], lw=0, color=disease, alpha=pop_fill_alpha)

ax.plot(label_loc, pop_avgs[1, :], color=healthy, lw=pop_lw)
# ax.plot(label_loc, samples[1, :, :5], color=disease, lw=1, alpha=0.5)
ax.fill(label_loc, pop_avgs[1, :], lw=0, color=healthy, alpha=pop_fill_alpha)

print(label_loc)
print(pop_avgs[0, :])
print(pop_avgs[1, :])
# Find population centroids
# centroid_θ, centroid_r = find_centroid(pop_avgs, label_loc)
# ax.plot(centroid_θ[0], centroid_r[0], 'o', color=disease, ms=cent_ms)
# ax.plot(centroid_θ[1], centroid_r[1], 'o', color=healthy, ms=cent_ms)

rotate_radar_labels(ax, y_offset=0.2)
format_ax(
    ax,
    figsize=radar_figsize,
    mar_l=0.175,
    mar_r=0.175,
    mar_t=0.175,
    mar_b=0.175,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, alias="flexible_dragonfly")

[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[0.98613193 2.18287618 4.67978488 4.54770221 1.68620714 0.98613193]
[5.         1.59486346 0.45811909 0.58168862 3.06966395 5.        ]


<PolarAxes: >

## Signature specification by competitors

In [16]:
fig, ax = prepare_radar_plot(label_loc, labels)

# Re-plot population averages
# ax.plot(label_loc, pop_avgs[0, :], color=disease, lw=pop_lw/2)
# ax.plot(label_loc, samples[0, :, :5], color=healthy, lw=1, alpha=0.5)
ax.fill(label_loc, pop_avgs[0, :], lw=0, color=disease, alpha=pop_fill_alpha / 2)
print(label_loc)
print(pop_avgs[0, :])

# ax.plot(label_loc, pop_avgs[1, :], color=healthy, lw=pop_lw/2)
# ax.plot(label_loc, samples[1, :, :5], color=disease, lw=1, alpha=0.5)
ax.fill(label_loc, pop_avgs[1, :], lw=0, color=healthy, alpha=pop_fill_alpha / 2)
print(label_loc)
print(pop_avgs[1, :])

# Indicate reference oligo concentrations
refs = pop_avgs.mean(0)
# ax.plot(np.tile(label_loc, (2,1)), pop_avgs, color='0.3', lw=pop_lw)
# ax.plot(np.tile(label_loc, (2,1)), pop_avgs, color='0.3', lw=pop_lw)


for loc, ref, avgs in zip(label_loc[:-1], refs, pop_avgs.T):
    inner = [0, ref]
    outer = [ref, r_max]
    inner_is_disease = (avgs[0] > avgs[1]).astype(int)
    inner_color = [disease, healthy][inner_is_disease]
    outer_color = [disease, healthy][~inner_is_disease]
    ax.plot([loc, loc], inner, color=inner_color, lw=1)
    ax.plot([loc, loc], outer, color=outer_color, lw=1)
    print(loc)


ax.plot(label_loc, refs, color="0.3", ls="none", marker="o", ms=cent_ms)
print(label_loc)
print(refs)


rotate_radar_labels(ax, y_offset=0.2)
format_ax(
    ax,
    figsize=radar_figsize,
    mar_l=0.175,
    mar_r=0.175,
    mar_t=0.175,
    mar_b=0.175,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, alias="transparent_eagle")

[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[0.98613193 2.18287618 4.67978488 4.54770221 1.68620714 0.98613193]
[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[5.         1.59486346 0.45811909 0.58168862 3.06966395 5.        ]
0.0
1.2566370614359172
2.5132741228718345
3.7699111843077517
5.026548245743669
[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[2.99306597 1.88886982 2.56895198 2.56469541 2.37793554 2.99306597]


<PolarAxes: >

## New patient assessment

In [17]:
new_patient_coeff = stats.uniform(0.5, 1.5).rvs([1, n_genes], random_state=SEED + 1)
new_patient = new_patient_coeff * np.roll(pop_avgs[0, :-1], 1)
new_patient = circularize(new_patient).squeeze()

fig, ax = prepare_radar_plot(label_loc, labels)


# ax.fill(label_loc, pop_avgs[0, :], lw=0, color=disease, alpha=pop_fill_alpha/2)
# ax.fill(label_loc, pop_avgs[1, :], lw=0, color=healthy, alpha=pop_fill_alpha/2)

for loc, ref, avgs in zip(label_loc[:-1], refs, pop_avgs.T):
    inner = [0, ref]
    outer = [ref, r_max]
    inner_is_disease = (avgs[0] > avgs[1]).astype(int)
    inner_color = [disease, healthy][inner_is_disease]
    outer_color = [disease, healthy][~inner_is_disease]
    ax.plot([loc, loc], inner, color=inner_color, lw=1)
    ax.plot([loc, loc], outer, color=outer_color, lw=1)
    print(loc)


# Indicate reference oligo concentrations
# ax.plot(label_loc, refs, color='0.3', ls='none', marker='o', ms=cent_ms)


ax.plot(label_loc, new_patient, color="0.2", lw=pop_lw + 1)
print(label_loc)
print(new_patient)

for loc, avgs, ref, pat in zip(label_loc[:-1], pop_avgs.T, refs, new_patient):
    closer_idx = np.argmin(np.abs(avgs - pat))
    color = [disease, healthy][closer_idx]
    #     ax.annotate('', xy=(loc, pat), xytext=(loc, ref),
    #                 arrowprops=dict(arrowstyle='-|>', lw=1, color=color, shrinkA=0, shrinkB=0, mutation_scale=6))
    # ax.plot([loc, loc], [ref, pat], color=color, lw=1)
    ax.plot([loc], [pat], color=color, ls="none", marker="o", ms=cent_ms * 3 / 4)
    print([loc])
    print([pat])


rotate_radar_labels(ax, y_offset=0.2)
format_ax(
    ax,
    figsize=radar_figsize,
    mar_l=0.175,
    mar_r=0.175,
    mar_t=0.175,
    mar_b=0.175,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, alias="silent_fox")

0.0
1.2566370614359172
2.5132741228718345
3.7699111843077517
5.026548245743669
[0.         1.25663706 2.51327412 3.76991118 5.02654825 6.28318531]
[1.94587014 0.53141599 2.89120578 5.39571516 5.14141248 1.94587014]
[0.0]
[1.9458701415433093]
[1.2566370614359172]
[0.5314159927100698]
[2.5132741228718345]
[2.891205784398892]
[3.7699111843077517]
[5.395715163700552]
[5.026548245743669]
[5.141412476437237]


<PolarAxes: >

# Architecture and serial dilution

### Import Data

In [19]:
cmax = 50

file = data_pth / "JG034 TMCC1 Gen2 Competitors - 59C v3.xlsx"
JG034 = (
    QuantStudio(file, "JG034")
    .import_data()
    .format_reactions()
    .index_reactions()
    .subtract_background()
    .normalize_reactions(cmax=cmax)  # , method='min-max')
    # .invert_fluorophore('HEX')
)

# for oligo in ['S036.5', 'S057.3.2', 'S057.4.2']:
#     JG071B.reactions.data['lg10 ' + oligo] = np.log10(JG071B.reactions.data[oligo])

## Serial Dilutions

In [21]:
this = JG034.reactions.data[
    (JG034.reactions.data["Cycle"] <= cmax)
    & (JG034.reactions.data.Target.str.contains("GC55"))
]
this.to_csv("Figure 1bii.csv")
wells = this.Well.unique()[::-2]

width = 3.80
height = 1.145

fig, axs = plt.subplots(1, len(wells), figsize=(width, height), sharey=True)

for well, ax in zip(wells, axs.flat):
    df = JG034.reactions.data[
        (JG034.reactions.data["Cycle"] <= cmax) & (JG034.reactions.data.Well == well)
    ]

    FAM = df[df.Target.str.contains("FAM")].Fluorescence.values
    HEX = df[df.Target.str.contains("HEX")].Fluorescence.values
    lg10_Copies = df.lg10_Copies.iloc[0]

    x = df.Cycle.unique()
    # ax.fill_between(x, FAM, HEX, color='0.7', alpha=0.5)
    ax.plot(x, FAM, lw=2, color=fam_color)
    ax.plot(x, HEX, lw=2, color=hex_color)

    print(x)
    print(FAM)
    print(HEX)

    y0 = HEX[-1] + 0.05 if FAM[-1] > HEX[-1] else HEX[-1] - 0.05
    y1 = FAM[-1] - HEX[-1] - 0.125 if FAM[-1] > HEX[-1] else FAM[-1] - HEX[-1] + 0.125
    # ax.arrow(x[-1], y0, 0, y1, color='k', head_width=3, head_length=0.1, length_includes_head=True, zorder=10)
    # Draw a dashed line from the arrow tip to the y-axis
    # ax.plot([x[-1], x[-1]], [0, min(HEX[-1], FAM[-1])-0.05], color='0.2', ls=':', lw=1, zorder=10)

    ax.set_ylim(0, 1.1)
    ax.set_xlim(-1, cmax + 11)
    ax.set_xticks(np.arange(0, 61, 20))
    ax.set_title(f"{lg10_Copies:.1f}")
    format_ax(
        ax,
        figsize=(width, height),
        mar_l=0.35,
        mar_r=0.05,
        mar_t=0.2,
        mar_b=0.32,
        ticklabelsize=6,
        labelsize=8,
        titlesize=10,
    )

axs[0].set_ylabel("Fluorescence")
axs[0].set_xlabel("Cycle")

#savefig(fig, alias="goldenrod_herring")

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]
[-0.00165428 -0.00147898 -0.00116772 -0.00075367 -0.00048171 -0.00012798
  0.00019094  0.00027437  0.00085348  0.00121229  0.00131267  0.00167293
  0.00204431  0.00189541  0.00245818  0.00309942  0.00332957  0.00326562
  0.00325431  0.00362634  0.00313408  0.00219022  0.00091503 -0.0005112
 -0.00144302 -0.00193875 -0.00141768 -0.00028013  0.00221021  0.00586463
  0.00987637  0.01379808  0.01688379  0.02030258  0.02341348  0.02653877
  0.02880473  0.03111402  0.03360056  0.0367122   0.03901216  0.04231012
  0.04526121  0.04722662  0.04968333  0.05225623  0.05494624  0.05792642
  0.06067184  0.06378738]
[-1.66567219e-03 -1.52386326e-03 -1.33168592e-03 -5.73488346e-04
 -3.29072642e-04  4.50041653e-04  6.20834790e-04  6.60918484e-04
  2.08551083e-04  2.93900902e-04  1.64391066e-03  1.72544044e-03
  1.82550943e-03  2.59435236e-03  3.174677

Text(0.5, 0, 'Cycle')

## Response Profile

In [22]:
import seaborn as sns

endpoints = (
    JG034.invert_fluorophore("HEX")
    .extract_endpoints(name="FAM-HEX")
    .invert_fluorophore("HEX")
)

width = 1.3
height = 1.145

this = JG034.endpoints[(JG034.endpoints.Target.str.contains("GC55"))]

wells = this.Well.unique()[::2]

x_var = "lg10_Copies"
hue = "FAM-HEX"
norm = mpl.colors.Normalize(vmin=-1, vmax=+1)

fig, ax = plt.subplots(1, 1, figsize=(width, height))

g = sns.scatterplot(
    data=this,
    x=x_var,
    y=hue,
    hue=hue,
    hue_norm=norm,
    palette=palette,
    s=spotsize,
    legend=False,
    ax=ax,
)
this.to_csv("Figure1biii.csv")

# spotsize **= 0.5
# ax.scatter(this[x_var], this[hue], cmap=palette, norm=norm, c=this[hue], s=spotsize)

ax.set_xlim(1, 9)
ax.set_ylim(-1, 1)
ax.set_xticks([2, 4, 6, 8])
ax.set_xlabel("log$_{10}$ WT copies")
ax.set_ylabel("FAM - HEX")
format_ax(
    ax,
    figsize=(width, height),
    mar_l=0.4,
    mar_r=0.05,
    mar_t=0.05,
    mar_b=0.32,
    ticklabelsize=6,
    labelsize=8,
    titlesize=10,
)

#savefig(fig, alias="chartreuse_iguana")

<Axes: xlabel='log$_{10}$ WT copies', ylabel='FAM - HEX'>

# Reaction components

In [24]:
JG069J = (
    QuantStudio(data_pth / "JG069J Final TB Experiment.xlsx", "JG069J")
    .import_data()
    .format_reactions()
    .index_reactions()
    .subtract_background()
    .normalize_reactions(cmax=cmax, method="min-max")
    # .invert_fluorophore('HEX')
)

In [26]:
width = 1.3
height = 1.145

for i, well in enumerate([343, 115]):
    fig, ax = plt.subplots(1, 1, figsize=(width, height))

    df = JG069J.reactions.data[
        (JG069J.reactions.data["Cycle"] <= cmax) & (JG069J.reactions.data.Well == well)
    ]
    df.to_csv(f'Figure1c{i}.csv')

    FAM = df[df.Target.str.contains("FAM")].Fluorescence.values * 4
    FAM -= FAM[0]
    HEX = df[df.Target.str.contains("HEX")].Fluorescence.values * 4
    HEX -= HEX[0]

    x = df.Cycle.unique()
    # ax.fill_between(x, FAM, HEX, color='0.7', alpha=0.5)
    ax.plot(x, FAM, lw=2, color=fam_color)
    ax.plot(x, HEX, lw=2, color=hex_color)

    print(x)
    print(FAM)
    print(HEX)

    x0 = x1 = x[-1] + 6
    y0 = HEX[-1]
    y1 = FAM[-1]
    ax.annotate(
        "",
        xytext=(x0, y0),
        xy=(x1, y1),
        color="k",
        arrowprops=dict(
            arrowstyle="-|>",
            color="k",
            lw=0.5,
            shrinkA=0,
            shrinkB=0,
            mutation_scale=8,
            mutation_aspect=1.5,
        ),
    )
    ax.annotate(
        "",
        xytext=(x0, y0),
        xy=(x1, y1),
        color="k",
        arrowprops=dict(
            arrowstyle="|-|", color="k", lw=0.5, shrinkA=0, shrinkB=0, mutation_scale=3
        ),
    )
    # Draw a dashed line from the arrow tip to the y-axis
    # ax.plot([x[-1], x[-1]], [0, min(HEX[-1], FAM[-1])-0.05], color='0.2', ls=':', lw=1, zorder=10)

    # ax.set_ylim(0, 1)
    ax.set_xlim(-1, cmax + 11)
    ax.set_ylim(-0.33, 7.5)
    ax.set_xticks(np.arange(0, 61, 20))
    # ax.set_title(well)
    ax.tick_params(axis="both", length=2, labelsize=16)

    ax.set_ylabel("Fluorescence", fontsize=16)
    ax.set_xlabel("Cycle", fontsize=16)

    format_ax(
        ax,
        figsize=(width, height),
        mar_l=0.4,
        mar_r=0.05,
        mar_t=0.05,
        mar_b=0.32,
        ticklabelsize=6,
        labelsize=8,
        titlesize=10,
    )

    alias = ["lavender_jackal", "glossy_newt"][i]

    #savefig(fig, alias=alias)

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]
[0.00000000e+00 1.27553647e-03 1.36182548e-03 4.05388851e-03
 4.82448018e-03 7.86185334e-03 1.68696555e-02 2.71016831e-02
 3.88249696e-02 4.16463121e-02 5.01795247e-02 5.68672312e-02
 5.68419608e-02 5.81718902e-02 5.81244312e-02 5.86064170e-02
 6.28784934e-02 6.77736998e-02 7.33949665e-02 8.02967005e-02
 9.48733797e-02 1.18759873e-01 1.60350097e-01 2.36968420e-01
 3.72517052e-01 5.98294712e-01 9.18447587e-01 1.33073432e+00
 1.79176941e+00 2.24555161e+00 2.66736865e+00 3.07064373e+00
 3.44768911e+00 3.83060707e+00 4.19657786e+00 4.52047907e+00
 4.84132380e+00 5.14395415e+00 5.41494170e+00 5.67232764e+00
 5.90599397e+00 6.09749763e+00 6.27128986e+00 6.42909151e+00
 6.56777644e+00 6.69280613e+00 6.82276909e+00 6.93096565e+00
 7.02709284e+00 7.12980141e+00]
[0.         0.01011629 0.01732697 0.01551619 0.02081698 0.02686939
 0.04605684 0.0

In [28]:
wells = this.Well.unique()[::-2]

dils = [[6, 1, 5], [0, 2, 4]]
weights = [[1.8, 1.1, 0.5], [2.0, 0.9, 0.6]]


width = 1.3 * 3 / 4
height = 1.145 * 3 / 4

# fig, axs = plt.subplots(2, 3, figsize = (width, height), sharey=True, sharex=True)

for row, (dil_row, weight_row) in enumerate(zip(dils, weights)):
    for col, (dil, weight) in enumerate(zip(dil_row, weight_row)):
        fig, ax = plt.subplots(1, 1, figsize=(width, height))

        well = wells[dil]

        df = JG034.reactions.data[
            (JG034.reactions.data["Cycle"] <= cmax)
            & (JG034.reactions.data.Well == well)
        ]

        df.to_csv(f"Figure1d{row, col}.csv")
        FAM = df[df.Target.str.contains("FAM")].Fluorescence.values * weight
        HEX = df[df.Target.str.contains("HEX")].Fluorescence.values * weight
        lg10_Copies = df.lg10_Copies.iloc[0]

        x = df.Cycle.unique()
        # ax.fill_between(x, FAM, HEX, color='0.7', alpha=0.5)
        ax.plot(x, FAM, lw=2, color=fam_color)
        ax.plot(x, HEX, lw=2, color=hex_color)

        print(x)
        print(FAM)
        print(HEX)

        ax.set_ylim(-0.1, 2)
        ax.set_xlim(-1, 61)
        ax.set_yticks([0, 1, 2])
        ax.set_xticks([0, 20, 40, 60])
        format_ax(
            ax,
            figsize=(width, height),
            mar_l=0.4,
            mar_r=0.05,
            mar_t=0.05,
            mar_b=0.32,
            ticklabelsize=6,
            labelsize=8,
            titlesize=10,
        )

        alias = [
            ["maroon_kangaroo", "teal_locust", "opaque_magpie"],
            ["bronze_ocelot", "silver_porcupine", "wooden_quail"],
        ][row][col]

        #savefig(fig, alias=alias)

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]
[-0.01877463 -0.01745258 -0.01599261 -0.01622077 -0.01461821 -0.01166307
 -0.0061798   0.00278139  0.01775686  0.04413621  0.09181103  0.17031065
  0.28572258  0.43664293  0.60590403  0.77283939  0.91919894  1.03566871
  1.11683633  1.17671662  1.21826467  1.25174621  1.27735956  1.29821694
  1.31651521  1.33438132  1.34888541  1.36299084  1.38159415  1.40024228
  1.41608009  1.43101458  1.44619645  1.45986762  1.47230862  1.48867909
  1.50803442  1.52511921  1.5387445   1.55415272  1.56769828  1.58242499
  1.60036184  1.61553542  1.63129384  1.6459588   1.66398577  1.68630072
  1.71189458  1.74016288]
[-0.00222059 -0.00219176 -0.0022103  -0.00033691 -0.00066703 -0.00202878
 -0.00179865  0.0012218   0.00225213  0.00356774  0.00429139  0.00786156
  0.01307946  0.02027077  0.02903504  0.04104263  0.0507585   0.05751219
  0.060504    0.0